# Feature Extraction

Here I extract audio features from the training files and the labeled soundscape segments, and save them to disk so I don't have to re-extract every time I want to train a model.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import time

from helpers import extract_features, extract_segment, FEATURE_NAMES

DATA_DIR = Path("../birdclef-2026")
FEAT_DIR = Path("../features")
FEAT_DIR.mkdir(exist_ok=True)

## 1. Sanity check on a few files

Let me first test the feature extraction on a few files to make sure it works and the output looks reasonable.

In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")
print(f"Total files: {len(train)}")
print(f"Feature names ({len(FEATURE_NAMES)}):")
print(FEATURE_NAMES[:5], "...", FEATURE_NAMES[-5:])

Total files: 35549
Feature names (60):
['mfcc1_mean', 'mfcc2_mean', 'mfcc3_mean', 'mfcc4_mean', 'mfcc5_mean'] ... ['chroma8_std', 'chroma9_std', 'chroma10_std', 'chroma11_std', 'chroma12_std']


In [3]:
# test on 3 files
for i in [0, 100, 1000]:
    row = train.iloc[i]
    path = DATA_DIR / "train_audio" / row['filename']
    feats = extract_features(str(path))
    print(f"{row['filename']}: shape={feats.shape}, min={feats.min():.4f}, max={feats.max():.4f}, any_nan={np.isnan(feats).any()}")

1161364/iNat1216197.ogg: shape=(60,), min=-343.1902, max=8715.0898, any_nan=False
22973/iNat870999.ogg: shape=(60,), min=-318.9165, max=7161.2699, any_nan=False
bafcur1/XC593408.ogg: shape=(60,), min=-406.6483, max=10820.8912, any_nan=False


## 2. Extract features for all train_audio

In [4]:
X_train = []
y_train = []
failed = []

start_time = time.time()

for idx, row in tqdm(train.iterrows(), total=len(train), desc="Extracting train_audio"):
    path = DATA_DIR / "train_audio" / row['filename']
    try:
        feats = extract_features(str(path))
        if np.isnan(feats).any():
            feats = np.nan_to_num(feats, 0.0)
        X_train.append(feats)
        y_train.append(str(row['primary_label']))
    except Exception as e:
        failed.append((row['filename'], str(e)))

elapsed = time.time() - start_time
print(f"\nDone in {elapsed/60:.1f} minutes")
print(f"Extracted: {len(X_train)}, Failed: {len(failed)}")

if failed:
    print("Failed files:")
    for f, e in failed[:10]:
        print(f"  {f}: {e}")

Extracting train_audio:   0%|▎                                                                                             | 124/35549 [00:16<2:18:02,  4.28it/s]/Users/macbook/Developer/Sorbonne M1/S2/ML/project/venv/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1536
  warnings.warn(
Extracting train_audio:   0%|▎                                                                                             | 136/35549 [00:17<1:11:38,  8.24it/s]/Users/macbook/Developer/Sorbonne M1/S2/ML/project/venv/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1115
  warnings.warn(
Extracting train_audio:   0%|▍                                                                                             | 176/35549 [00:22<1:28:56,  6.63it/s]/Users/macbook/Developer/Sorbonne M1/S2/ML/project/venv/lib/python3.13/site-packages/librosa/core/spectrum.py:266: 


Done in 120.5 minutes
Extracted: 35549, Failed: 0


In [5]:
X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

np.save(FEAT_DIR / "X_train.npy", X_train)
np.save(FEAT_DIR / "y_train.npy", y_train)
print("Saved to features/")

X_train shape: (35549, 60)
y_train shape: (35549,)
Saved to features/


## 3. Extract features for labeled soundscape segments

The soundscape labels are multi-label (multiple species per segment, separated by `;`). For now I'm just storing the raw label string as-is. In the baseline I'll simplify things by treating it as a single label, and maybe later try proper multi-label classification.

In [6]:
sc_labels = pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv")
print(f"Labeled segments: {len(sc_labels)}")
sc_labels.head(3)

Labeled segments: 1478


,filename,start,end,primary_label
0,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:00,00:00:05,22961;23158;24321;517063;65380
1,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:05,00:00:10,22961;23158;24321;517063;65380
2,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:10,00:00:15,22961;23158;24321;517063;65380


In [7]:
def parse_time(t):
    parts = t.split(':')
    return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])

X_sc = []
y_sc = []
failed_sc = []

start_time = time.time()

for idx, row in tqdm(sc_labels.iterrows(), total=len(sc_labels), desc="Extracting soundscapes"):
    path = DATA_DIR / "train_soundscapes" / row['filename']
    start_sec = parse_time(row['start'])
    end_sec = parse_time(row['end'])
    try:
        feats = extract_segment(str(path), start_sec, end_sec)
        if np.isnan(feats).any():
            feats = np.nan_to_num(feats, 0.0)
        X_sc.append(feats)
        y_sc.append(row['primary_label'])
    except Exception as e:
        failed_sc.append((row['filename'], str(e)))

elapsed = time.time() - start_time
print(f"\nDone in {elapsed/60:.1f} minutes")
print(f"Extracted: {len(X_sc)}, Failed: {len(failed_sc)}")

if failed_sc:
    print("Failed:")
    for f, e in failed_sc[:10]:
        print(f"  {f}: {e}")

Extracting soundscapes:   2%|█▌                                                                                                | 23/1478 [00:00<00:40, 35.75it/s]/Users/macbook/Developer/Sorbonne M1/S2/ML/project/venv/lib/python3.13/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
Extracting soundscapes: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 1478/1478 [00:41<00:00, 35.78it/s]


Done in 0.7 minutes
Extracted: 1478, Failed: 0


In [8]:
X_sc = np.array(X_sc)
y_sc = np.array(y_sc)

print(f"X_soundscape shape: {X_sc.shape}")
print(f"y_soundscape shape: {y_sc.shape}")

np.save(FEAT_DIR / "X_soundscape.npy", X_sc)
np.save(FEAT_DIR / "y_soundscape.npy", y_sc)
print("Saved to features/")

X_soundscape shape: (1478, 60)
y_soundscape shape: (1478,)
Saved to features/


## 4. Quick check of saved features

In [9]:
# reload and verify
X = np.load(FEAT_DIR / "X_train.npy")
y = np.load(FEAT_DIR / "y_train.npy", allow_pickle=True)
print(f"Train: X={X.shape}, y={y.shape}, unique labels={len(np.unique(y))}")
print(f"  min={X.min():.4f}, max={X.max():.4f}, NaNs={np.isnan(X).sum()}")

X2 = np.load(FEAT_DIR / "X_soundscape.npy")
y2 = np.load(FEAT_DIR / "y_soundscape.npy", allow_pickle=True)
print(f"\nSoundscape: X={X2.shape}, y={y2.shape}")
print(f"  min={X2.min():.4f}, max={X2.max():.4f}, NaNs={np.isnan(X2).sum()}")

Train: X=(35549, 60), y=(35549,), unique labels=206
  min=-955.3348, max=15109.3750, NaNs=0

Soundscape: X=(1478, 60), y=(1478,)
  min=-496.7241, max=13243.3107, NaNs=0
